# World Cup 2026 — Player Statistics Pipeline

Run the cells **in order** the first time. After that you can re-run any single cell independently.

| Cell | What it does | API calls |
|------|--------------|-----------|
| 1. Configuration | Set your API key, season | — |
| 2. Imports | Load all modules | — |
| 3. Init DB | Create DuckDB schema | — |
| 4. Fetch Teams | 32 WC nations | 1 |
| 5. Fetch Squads | ~736 players (basic info) | ~32 |
| 6. Enrich Players | Club team, nationality, height/weight per WC player | ~736 |
| 7. Fetch Match Stats | All matches for clubs with WC players | ~1 per match |
| 8. DB Check | Row counts (optional) | — |


## Cell 1 — Configuration
Edit the values here before running anything else.

In [ ]:
# ── Your API-Football key from https://api-sports.io ─────────────────────────
API_FOOTBALL_KEY = "your_api_key_here"

# ── Local database file ───────────────────────────────────────────────────────
DB_PATH = "playerstats.db"

# ── Club season to fetch match stats for ─────────────────────────────────────
# European leagues finishing in 2026 are season 2025 (e.g. 2025-26 PL)
SEASON = 2025

# ── Rate limit (requests per minute) ─────────────────────────────────────────
# Free tier = 10/min.  Paid tiers can go higher.
REQUESTS_PER_MINUTE = 10

print(f"Season : {SEASON}")
print(f"DB path: {DB_PATH}")

## Cell 2 — Imports

In [ ]:
import logging

from rich.console import Console
from rich.logging import RichHandler
from rich.panel import Panel
from rich.table import Table

from src.api.client import FootballAPIClient
from src.db.database import Database
from src.etl.extract import Extractor
from src.etl.load import Loader
from src.etl.transform import (
    enrich_players_from_stats,
    generate_date_dimension,
    transform_competitions,
    transform_competitions_from_fixtures,
    transform_match_player_stats,
    transform_matches,
    transform_players,
    transform_teams,
    transform_teams_from_fixtures,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[RichHandler(rich_tracebacks=True, show_path=False)],
)
console = Console()


def _make_db():
    return Database(DB_PATH)


def _make_client():
    if not API_FOOTBALL_KEY or API_FOOTBALL_KEY == "your_api_key_here":
        raise ValueError("Set API_FOOTBALL_KEY in Cell 1 before running.")
    return FootballAPIClient(api_key=API_FOOTBALL_KEY, requests_per_minute=REQUESTS_PER_MINUTE)


def _summary(title, data):
    t = Table(title=title, show_header=False, expand=False)
    t.add_column("Key", style="cyan")
    t.add_column("Value", style="white")
    for k, v in data.items():
        t.add_row(str(k), str(v))
    console.print(t)


print("Imports OK.")

## Cell 3 — Initialise Database
Creates all tables and views in the local DuckDB file. Safe to re-run.

In [ ]:
console.print(Panel("[bold]Initialising database schema[/bold]", style="blue"))

with _make_db() as db:
    db.initialize_schema()
    loader = Loader(db)
    n = loader.load_date_dimension(generate_date_dimension(2024, 2027))

console.print(f"[green]Done.[/green] {n} date rows loaded.")

## Cell 4 — Fetch World Cup 2026 Teams
Downloads the 32 national teams. **API calls: 1**

In [ ]:
console.print(Panel("[bold]Fetching World Cup 2026 teams[/bold]", style="blue"))

client = _make_client()
extractor = Extractor(client)
raw_teams = extractor.extract_world_cup_teams(season=2026)
team_rows = transform_teams(raw_teams)

with _make_db() as db:
    n = Loader(db).upsert_teams(team_rows)

_summary("Step 4 — Teams", {
    "Teams fetched": len(raw_teams),
    "Rows upserted": n,
    "API calls": client._request_count,
})

## Cell 5 — Fetch Squad Rosters
Downloads the player list for every World Cup team. Gives basic info (name, age, position) but not club team yet — that comes in Cell 6.

**API calls: ~1 per team (~32 total)**

In [ ]:
console.print(Panel("[bold]Fetching World Cup squads[/bold]", style="blue"))

with _make_db() as db:
    teams_df = db.query("SELECT api_team_id, name FROM dim_team WHERE is_world_cup_2026 = TRUE")

if teams_df.empty:
    print("No WC teams in DB — run Cell 4 first.")
else:
    raw_stubs = [{"team": {"id": int(r["api_team_id"]), "name": r["name"]}} for _, r in teams_df.iterrows()]

    client = _make_client()
    extractor = Extractor(client)
    raw_squads = extractor.extract_world_cup_squads(teams=raw_stubs)

    with _make_db() as db:
        loader = Loader(db)
        player_rows = transform_players(raw_squads, wc_team_map=loader.get_team_id_map())
        n = loader.upsert_players(player_rows)

    _summary("Step 5 — Squads", {
        "Teams processed": len(raw_squads),
        "Players upserted": n,
        "API calls": client._request_count,
    })

## Cell 6 — Enrich Players
For each WC player, fetches their full profile for the given season:
- **Club team** (which club they play for — needed for Cell 7)
- Nationality, height, weight, birth date

**API calls: 1 per WC player (~736 total)**

> On the free tier (100 calls/day) run this over multiple days, or upgrade to a paid plan.

In [ ]:
console.print(Panel(f"[bold]Enriching WC players (season {SEASON})[/bold]", style="blue"))

with _make_db() as db:
    players_df = db.query("SELECT api_player_id FROM dim_player WHERE world_cup_team_id IS NOT NULL")

if players_df.empty:
    print("No WC players in DB — run Cell 5 first.")
else:
    player_ids = [int(x) for x in players_df["api_player_id"].tolist()]
    print(f"Enriching {len(player_ids)} players...")

    client = _make_client()
    extractor = Extractor(client)
    raw_stats = extractor.extract_player_stats(player_ids=player_ids, seasons=[SEASON])

    with _make_db() as db:
        loader = Loader(db)

        # Load competitions and club teams referenced in the stats
        loader.upsert_competitions(transform_competitions(raw_stats))
        existing_team_map = loader.get_team_id_map()
        new_clubs, seen = [], set(existing_team_map.keys())
        for record in raw_stats:
            for stat in record.get("statistics", []):
                t = stat.get("team", {})
                aid = t.get("id")
                if aid and aid not in seen:
                    seen.add(aid)
                    new_clubs.append({"api_team_id": aid, "name": t.get("name", "?"),
                                      "short_name": None, "country": None,
                                      "logo_url": t.get("logo"), "is_world_cup_2026": False})
        if new_clubs:
            loader.upsert_teams(new_clubs)

        # Enrich player rows with bio + club_team_id
        ep_df = db.query("SELECT api_player_id, world_cup_team_id FROM dim_player")
        existing = {
            int(r["api_player_id"]): {"world_cup_team_id": int(r["world_cup_team_id"]) if r.get("world_cup_team_id") is not None else None}
            for _, r in ep_df.iterrows()
        }
        enriched = enrich_players_from_stats(raw_stats, existing)
        n = loader.upsert_players(enriched)

    _summary("Step 6 — Enrich Players", {
        "Players processed": len(player_ids),
        "Stat records fetched": len(raw_stats),
        "Players updated": n,
        "API calls": client._request_count,
    })

## Cell 7 — Fetch Per-Match Stats (WC Players Only)
Now that we know which club each WC player belongs to, we:
1. Find all unique clubs that employ at least one WC player
2. Fetch all completed fixtures for each club
3. Fetch per-match player stats for each fixture
4. Store **only WC players** in the fact table (others are ignored automatically)

Fixtures shared between two clubs (e.g. PSG vs Man City) are fetched only once.

**API calls: 1 per club (fixture list) + 1 per completed match**

In [ ]:
console.print(Panel(f"[bold]Fetching WC player match stats (season {SEASON})[/bold]", style="blue"))

with _make_db() as db:
    clubs_df = db.query("""
        SELECT DISTINCT ct.api_team_id, ct.name
        FROM dim_player p
        JOIN dim_team ct ON p.club_team_id = ct.team_id
        WHERE p.world_cup_team_id IS NOT NULL
          AND p.club_team_id IS NOT NULL
    """)

if clubs_df.empty:
    print("No club teams found — run Cell 6 (enrich-players) first.")
else:
    wc_clubs = [{"api_team_id": int(r["api_team_id"]), "name": r["name"]} for _, r in clubs_df.iterrows()]
    print(f"{len(wc_clubs)} unique clubs with WC players found.")

    client = _make_client()
    extractor = Extractor(client)
    enriched_fixtures = extractor.extract_wc_player_match_stats(wc_club_teams=wc_clubs, season=SEASON)

    if not enriched_fixtures:
        print("No completed fixtures found.")
    else:
        with _make_db() as db:
            loader = Loader(db)

            loader.upsert_competitions(transform_competitions_from_fixtures(enriched_fixtures))

            existing_map = loader.get_team_id_map()
            new_teams = [t for t in transform_teams_from_fixtures(enriched_fixtures)
                         if t["api_team_id"] not in existing_map]
            if new_teams:
                loader.upsert_teams(new_teams)

            team_id_map = loader.get_team_id_map()
            comp_id_map = loader.get_competition_id_map()

            n_matches = loader.upsert_matches(
                transform_matches(enriched_fixtures, team_id_map, comp_id_map)
            )
            n_facts = loader.upsert_match_player_stats(
                transform_match_player_stats(
                    enriched_fixtures,
                    player_id_map=loader.get_player_id_map(),
                    match_id_map=loader.get_match_id_map(),
                    team_id_map=team_id_map,
                    competition_id_map=comp_id_map,
                    national_team_map=loader.get_national_team_map(),
                )
            )

        _summary("Step 7 — Match Stats", {
            "Season": SEASON,
            "WC clubs processed": len(wc_clubs),
            "Fixtures loaded": len(enriched_fixtures),
            "Matches upserted": n_matches,
            "Fact rows (WC players only)": n_facts,
            "API calls": client._request_count,
        })

## Cell 8 — DB Check (optional)
Run at any time to see how much data is loaded.

In [ ]:
with _make_db() as db:
    counts = {
        "dim_team (WC nations)": db.query("SELECT COUNT(*) AS n FROM dim_team WHERE is_world_cup_2026").iloc[0]["n"],
        "dim_team (all clubs)": db.query("SELECT COUNT(*) AS n FROM dim_team").iloc[0]["n"],
        "dim_player (WC players)": db.query("SELECT COUNT(*) AS n FROM dim_player WHERE world_cup_team_id IS NOT NULL").iloc[0]["n"],
        "dim_player (with club assigned)": db.query("SELECT COUNT(*) AS n FROM dim_player WHERE club_team_id IS NOT NULL").iloc[0]["n"],
        "dim_match": db.query("SELECT COUNT(*) AS n FROM dim_match").iloc[0]["n"],
        "fact_player_match_stats": db.query("SELECT COUNT(*) AS n FROM fact_player_match_stats").iloc[0]["n"],
    }

_summary("Row counts", counts)